### `2.) Feature Engineering`

In this stage we construct additional variables that capture borrower risk characteristics more effectively than the raw dataset.

Credit risk models often benefit from engineered features such as borrower leverage, credit utilization, and summarized credit scores.

These variables help the model better represent borrower financial capacity and repayment risk.

LOAN CLEAN DATASET

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("clean_lendingclub.parquet")

df.shape

(1345310, 21)

### Creating Average FICO Score

The dataset contains two FICO variables:

- `fico_range_low`
- `fico_range_high`

These represent a range of possible credit scores for the borrower.

We compute the **Average FICO score** to obtain a single representative credit score variable.

In [5]:
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

### Loan-to-Income Ratio

The Loan-to-Income ratio measures the size of the loan relative to borrower income.

Higher values indicate that the borrower is taking on a larger loan compared to their income, which may increase default risk.

In [6]:
df["loan_income_ratio"] = df["loan_amnt"] / df["annual_inc"]

### Credit Card Utilization Proxy

Revolving balance represents outstanding credit card debt.

We scale it relative to borrower income to approximate the borrower's credit utilization pressure.

In [7]:
df["revol_income_ratio"] = df["revol_bal"] / df["annual_inc"]

### Simplifying FICO Variables

Since we now use the average FICO score, the original FICO range variables are no longer required.

So, we Drop Original FICO Columns

In [8]:
df.drop(["fico_range_low", "fico_range_high"], axis=1, inplace=True)

Quick Check the Dataframe

In [6]:
df.head()
# df.head().to_csv("sample_head.csv", index=False)

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,dti,open_acc,revol_bal,revol_util,total_acc,recoveries,default,fico_avg,loan_income_ratio,revol_income_ratio
0,3600.0,36,13.99,123.03,C,C4,10.0,MORTGAGE,55000.0,Not Verified,...,5.91,7.0,2765.0,29.7,13.0,0.0,0,677.0,0.065455,0.050273
1,24700.0,36,11.99,820.28,C,C1,10.0,MORTGAGE,65000.0,Not Verified,...,16.06,22.0,21470.0,19.2,38.0,0.0,0,717.0,0.380000,0.330308
2,20000.0,60,10.78,432.66,B,B4,10.0,MORTGAGE,63000.0,Not Verified,...,10.78,6.0,7869.0,56.2,18.0,0.0,0,697.0,0.317460,0.124905
4,10400.0,60,22.45,289.91,F,F1,3.0,MORTGAGE,104433.0,Source Verified,...,25.37,12.0,21929.0,64.5,35.0,0.0,0,697.0,0.099585,0.209982
5,11950.0,36,13.44,405.18,C,C3,4.0,RENT,34000.0,Source Verified,...,10.20,5.0,8822.0,68.4,6.0,0.0,0,692.0,0.351471,0.259471


### Encoding Categorical Variables

Several variables in the dataset are categorical, meaning they contain text labels rather than numeric values. Examples include:

- loan grade
- home ownership status
- loan purpose
- verification status

Machine learning algorithms require numeric inputs, so we convert these categorical variables into binary indicator variables using **one-hot encoding**.

Each category becomes a separate column indicating whether that category applies to a given loan.

IDENTIFY CATEGORICAL VARIABLES

In [10]:
cat_cols = [
"grade",
"sub_grade",
"home_ownership",
"verification_status",
"purpose"
]

ONE-HOT ENCODE THE CATEGORICAL VARIABLES

In [11]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
### drop_first=True prevents dummy variable trap in regression models.

In [12]:
df.shape

(1345310, 77)

### Result of One-Hot Encoding

Each categorical variable has been converted into multiple binary indicator columns.

For example, the loan grade variable is transformed into columns such as:

- grade_B
- grade_C
- grade_D
- grade_E

A value of 1 indicates the loan belongs to that category, while 0 indicates otherwise.

This transformation allows machine learning models to incorporate categorical information when estimating borrower default risk.

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1345310 entries, 0 to 2260697
Data columns (total 77 columns):
 #   Column                               Non-Null Count    Dtype         
---  ------                               --------------    -----         
 0   loan_amnt                            1345310 non-null  float64       
 1   term                                 1345310 non-null  int32         
 2   int_rate                             1345310 non-null  float64       
 3   installment                          1345310 non-null  float64       
 4   emp_length                           1345310 non-null  float64       
 5   annual_inc                           1345310 non-null  float64       
 6   issue_d                              1345310 non-null  datetime64[ns]
 7   dti                                  1345310 non-null  float64       
 8   open_acc                             1345310 non-null  float64       
 9   revol_bal                            1345310 non-null  float64

SAVE CLEANED DATASET

In [14]:
df.to_parquet("lendingclub_features.parquet", index=False)